# Analítica predictiva de supervivencia en pasajeros del Titanic

Este cuaderno desarrolla un flujo completo de analítica de datos sobre el archivo `tested.csv`. El objetivo es entender la estructura del conjunto de datos, evaluar su calidad, explorar patrones relevantes y construir un modelo de clasificación para predecir la variable `Survived`.

El trabajo está organizado como un proyecto profesional de datos: carga, diccionario de variables, limpieza, análisis exploratorio, preparación de variables, entrenamiento, evaluación e interpretación.

## 1. Contexto del conjunto de datos

Cada fila representa un pasajero. Las variables combinan información demográfica, socioeconómica y de viaje. La variable objetivo es `Survived`, codificada como `1` para supervivencia y `0` para no supervivencia.

| Variable | Descripción | Tipo esperado |
|---|---|---|
| `PassengerId` | Identificador único del pasajero | Numérico |
| `Survived` | Variable objetivo: 0 = no sobrevivió, 1 = sobrevivió | Categórica/binaria |
| `Pclass` | Clase del boleto: 1, 2 o 3 | Categórica ordinal |
| `Name` | Nombre completo del pasajero | Texto |
| `Sex` | Sexo registrado del pasajero | Categórica |
| `Age` | Edad en años | Numérica |
| `SibSp` | Número de hermanos/cónyuges a bordo | Numérica discreta |
| `Parch` | Número de padres/hijos a bordo | Numérica discreta |
| `Ticket` | Código del boleto | Texto |
| `Fare` | Tarifa pagada | Numérica |
| `Cabin` | Cabina asignada | Texto/categórica |
| `Embarked` | Puerto de embarque | Categórica |

In [ ]:
# 2. Importar librerías
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", 50)

## 3. Carga de datos

La siguiente celda busca `tested.csv` en la misma carpeta del cuaderno. Si el notebook se ejecuta desde otra ubicación, también revisa el directorio de trabajo actual.

In [ ]:
# 3. Cargar el archivo CSV
notebook_dir = Path.cwd()
csv_candidates = [notebook_dir / "tested.csv", Path("tested.csv")]
csv_path = next((path for path in csv_candidates if path.exists()), None)

if csv_path is None:
    raise FileNotFoundError("No se encontró tested.csv. Ubícalo junto al cuaderno o en el directorio de trabajo.")

df = pd.read_csv(csv_path)
print(f"Archivo cargado: {csv_path.resolve()}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head()

## 4. Tabla inicial de datos

Antes de modelar, conviene inspeccionar registros reales. Esta tabla permite verificar nombres de columnas, formatos, valores faltantes visibles y posibles variables que no deben entrar directamente al modelo.

In [ ]:
# Muestra estructurada de los primeros pasajeros
df.head(10)

## 5. Calidad del dato

En esta etapa revisamos tipos de datos, completitud y duplicados. Esta revisión define las decisiones de limpieza: imputar edades y tarifas, transformar variables categóricas y excluir campos de identificación o texto libre cuando no aporten señal directa al modelo.

In [ ]:
# Estructura y tipos de datos
df.info()

# Valores faltantes por columna
missing_summary = (
    df.isna()
    .sum()
    .to_frame("faltantes")
    .assign(porcentaje=lambda x: (x["faltantes"] / len(df) * 100).round(2))
    .sort_values("faltantes", ascending=False)
)
missing_summary

In [ ]:
# Duplicados y estadísticos descriptivos
print(f"Filas duplicadas: {df.duplicated().sum()}")
df.describe(include="all").transpose()

## 6. Análisis exploratorio

El análisis exploratorio busca responder tres preguntas:

1. ¿Cómo se distribuye la variable objetivo?
2. ¿Qué variables se relacionan más con la supervivencia?
3. ¿Qué limitaciones del archivo debemos tener presentes antes de interpretar el modelo?

In [ ]:
# Distribución de la variable objetivo
survival_distribution = (
    df["Survived"]
    .value_counts(normalize=True)
    .rename(index={0: "No sobrevivió", 1: "Sobrevivió"})
    .mul(100)
    .round(2)
    .to_frame("porcentaje")
)
survival_distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df, x="Survived", ax=axes[0])
axes[0].set_title("Distribución de supervivencia")
axes[0].set_xticklabels(["No", "Sí"])
axes[0].set_xlabel("Sobrevivió")
axes[0].set_ylabel("Pasajeros")

sns.barplot(data=df, x="Sex", y="Survived", ci=None, ax=axes[1])
axes[1].set_title("Tasa de supervivencia por sexo")
axes[1].set_xlabel("Sexo")
axes[1].set_ylabel("Tasa de supervivencia")

sns.barplot(data=df, x="Pclass", y="Survived", ci=None, ax=axes[2])
axes[2].set_title("Tasa de supervivencia por clase")
axes[2].set_xlabel("Clase")
axes[2].set_ylabel("Tasa de supervivencia")

plt.tight_layout()
plt.show()

In [ ]:
# Cruces analíticos clave
pd.crosstab(df["Sex"], df["Survived"], margins=True, normalize="index").round(3)

> **Nota analítica importante:** en este archivo, `Survived` está completamente determinado por `Sex`: todos los registros femeninos tienen `Survived = 1` y todos los masculinos tienen `Survived = 0`. Esto no invalida el ejercicio técnico, pero sí exige cautela: el modelo aprenderá una regla casi perfecta basada en una sola variable, por lo que sus métricas pueden parecer excelentes sin representar necesariamente un modelo generalizable a otros datos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(data=df, x="Age", hue="Survived", bins=25, kde=True, ax=axes[0])
axes[0].set_title("Distribución de edad por supervivencia")
axes[0].set_xlabel("Edad")

sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[1])
axes[1].set_title("Tarifa pagada según supervivencia")
axes[1].set_xlabel("Sobrevivió")
axes[1].set_ylabel("Tarifa")
axes[1].set_xticklabels(["No", "Sí"])

plt.tight_layout()
plt.show()

## 7. Preparación de variables

Para construir el modelo se excluyen `PassengerId`, `Name`, `Ticket` y `Cabin`. Las tres primeras son identificadores o texto libre con baja utilidad directa en esta versión del análisis. `Cabin` tiene demasiados valores faltantes, por lo que se descarta para evitar una imputación poco confiable.

Las variables numéricas se imputan con la mediana y se escalan. Las categóricas se imputan con la categoría más frecuente y se codifican con `OneHotEncoder`.

In [ ]:
target = "Survived"
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = df[features].copy()
y = df[target].copy()

numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

## 8. Entrenamiento del modelo

Se utiliza regresión logística porque es un modelo interpretable, apropiado para clasificación binaria y útil como línea base en proyectos de analítica predictiva.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(X_train, y_train)
print("Modelo entrenado correctamente.")

## 9. Evaluación

La evaluación combina métricas de clasificación, matriz de confusión y AUC. Dado el patrón detectado entre `Sex` y `Survived`, estas métricas deben interpretarse como validación del flujo técnico y no como garantía de desempeño en un conjunto de datos distinto.

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["No sobrevivió", "Sobrevivió"]))
print(f"AUC ROC: {roc_auc_score(y_test, y_proba):.3f}")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Sí"])
disp.plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

## 10. Interpretación de variables

Para interpretar el modelo, extraemos los coeficientes de la regresión logística. Los valores positivos empujan la predicción hacia supervivencia y los negativos hacia no supervivencia.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

importance = (
    pd.DataFrame({"variable": feature_names, "coeficiente": coefficients})
    .assign(abs_coeficiente=lambda x: x["coeficiente"].abs())
    .sort_values("abs_coeficiente", ascending=False)
)

importance.head(12)

In [ ]:
plt.figure(figsize=(10, 5))
top_importance = importance.head(10).sort_values("coeficiente")
sns.barplot(data=top_importance, x="coeficiente", y="variable", palette="coolwarm")
plt.axvline(0, color="black", linewidth=1)
plt.title("Variables con mayor influencia en el modelo")
plt.xlabel("Coeficiente")
plt.ylabel("Variable transformada")
plt.tight_layout()
plt.show()

## 11. Predicción con nuevos pasajeros

El siguiente ejemplo muestra cómo usar el pipeline completo para estimar la probabilidad de supervivencia de nuevos registros. El modelo recibe datos en el mismo formato de las variables seleccionadas.

In [ ]:
new_passengers = pd.DataFrame(
    [
        {"Pclass": 1, "Sex": "female", "Age": 29, "SibSp": 0, "Parch": 0, "Fare": 85.0, "Embarked": "C"},
        {"Pclass": 3, "Sex": "male", "Age": 41, "SibSp": 1, "Parch": 0, "Fare": 8.0, "Embarked": "S"},
    ]
)

predictions = new_passengers.copy()
predictions["probabilidad_supervivencia"] = model.predict_proba(new_passengers)[:, 1].round(3)
predictions["prediccion"] = model.predict(new_passengers)
predictions["prediccion"] = predictions["prediccion"].map({0: "No sobrevivió", 1: "Sobrevivió"})
predictions

## 12. Conclusiones

- El archivo contiene 418 pasajeros y 12 variables originales.
- Las columnas con mayor ausencia son `Cabin` y `Age`; por eso se descarta `Cabin` y se imputa `Age` con una estrategia robusta.
- La variable `Sex` domina el comportamiento de `Survived` en este archivo. Esto permite un desempeño predictivo muy alto, pero también revela una limitación importante del dataset.
- El pipeline construido es reproducible: integra imputación, escalado, codificación categórica, entrenamiento y predicción.
- Para un proyecto real, el siguiente paso sería validar con un conjunto de datos independiente y enriquecer variables como título del pasajero, tamaño familiar, cubierta de cabina y grupos de edad.